# Max $Q_i$ value for a given mode and a given frequency (from Pozar)

Formula from Pozar: Chapter 6.4 *Circular Waveguid Cavity Resonators*

$Q_i = \dfrac{(kR)^3 \eta R L}{4(\alpha_{nl}')^2 R_S} \times \dfrac{1 - \left(\dfrac{n}{\alpha_{nl}'}\right)^2}{\dfrac{RL}{2}\left(1 + \left(\dfrac{\beta Rn}{(\alpha_{nl}')^2}\right)^2\right) + \left(\dfrac{\beta R^2}{\alpha_{nl}'}\right)^2\left(1 - \dfrac{n^2}{(\alpha_{nl}')^2}\right)}$

with:
- $R$ the radius of the cavity
- $L$ the length of the cavity
- $TE_{nlm}$ the electric mode
- $k=\dfrac{2\pi f}{c}$
- $\eta = \sqrt{\dfrac{\mu}{\varepsilon}}$
    - $\mu = \mu_0\mu_r$
    - $\varepsilon = \varepsilon_0\varepsilon_r$
- $\alpha_{nl}'$ the l-th zero of the derivative of the n-th order Bessel function of the first kind $J'_n$
- $R_S=\dfrac{1}{\sigma\delta}$ the surface resistivity
    - $\sigma$ the conductivity
    - $\delta = \dfrac{1}{\sqrt{\pi f\mu_0\sigma}}$ the skin depth
- $\beta=\dfrac{m\pi}{L}$

For a cylindrical cavity, we also have :

$f_{nlm} = \dfrac{c}{2\pi}\sqrt{\left(\dfrac{\alpha_{nl}'}{R}\right)^2 + \left(\dfrac{m\pi}{L}\right)^2}$

In [31]:
import numpy as np
from scipy.special import jnp_zeros
import scipy.constants as cst
import re

In [32]:
mode = 'TE115'
f0 = 9.3e9
R = 25e-3
c = 299792458

In [33]:
def indices(mode):

    match = re.match(r'^TE(\d+)$', mode)
    if match:
        ints = match.group(1)
        if len(ints) >= 3:
            n = int(ints[0])
            l = int(ints[1])
            m = int(ints[2:])
            return n, l, m

    raise ValueError(f"Format de mode non reconnu : '{mode_str}'. Attendu : 'TE114', 'TE 1 1 4', etc.")

In [34]:
def length(mode, f0, R):
    
    n, l, m = indices(mode)
    alpha = jnp_zeros(n, l)[-1]
    if (2*np.pi*f0 / cst.c)**2 < (alpha/R)**2:
        raise ValueError(f'Impossible to reach this frequency with {mode}')
    
    length = m*np.pi / np.sqrt((2*np.pi*f0 / c)**2 - (alpha/R)**2)
    
    return length
    

In [45]:
def compute_Q(mode, f0, R):
    
    n, l, m = indices(mode)
    alpha = jnp_zeros(n, l)[-1]
    L = length(mode, f0, R)
    
    k = 2*np.pi*f0 / cst.c
    eta = np.sqrt(cst.mu_0 / cst.epsilon_0)
    sigma = 59.6e6
    delta = 1 / np.sqrt(np.pi * f0 * cst.mu_0 * sigma)
    Rs = 1 / (sigma * delta)
    beta = m * np.pi / L
    
    A = (((k * R)**3) * eta * R * L) / (4 * (alpha**2) * Rs)
    B = 1 - (n / alpha)**2
    C1 = (R * L / 2) * (1 + (beta * R * n / (alpha**2))**2)
    C2 = (beta * R**2 / alpha)**2 * (1 - (n / alpha)**2)
    
    Q = A * B / (C1 + C2)
    
    return Q
    

In [69]:
Q_i = compute_Q('TE114', 6.845e9, R)
print(f'Q_i = {Q_i:.0f}')
print(f'Q_max = {Q_i/2:.0f} Critical regime')

Q_i = 30427
Q_max = 15214 Critical regime
